In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from ast import literal_eval
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import math

# 1. 读取数据
df = pd.read_csv('hardness_curve_dataset.csv')

# 2. 解析 hv_data 为数值数组
df['hv_data'] = df['hv_data'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
hv_array = np.array(df['hv_data'].tolist())

# 3. 提取输入特征
feature_columns = ["Image_Name", 'C', 'Si', 'Mn', 'P', 'Cr', 'Perimeter', 'Aspect_Ratio',
                   'Compactness', 'Density', 'Uniformity', 'carburizing_temp', 'temp_gradient']
X = df[feature_columns[1:]].values
Y = hv_array

# 4. 划分数据集（训练+验证、测试）
X_trainval, X_test, y_trainval, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
# 再划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42)

# 5. 特征标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 转换为numpy数组
X_train_np = X_train.astype(np.float32)
X_val_np = X_val.astype(np.float32)
X_test_np = X_test.astype(np.float32)
y_train_np = y_train.astype(np.float32)
y_val_np = y_val.astype(np.float32)
y_test_np = y_test.astype(np.float32)

# 准备LSTM输入序列
def create_sequence_inputs(X_array, seq_len):
    n_samples, n_features = X_array.shape
    X_seq = np.zeros((n_samples, seq_len, n_features + 1), dtype=np.float32)
    for i in range(n_samples):
        for t in range(seq_len):
            X_seq[i, t, :-1] = X_array[i]
            X_seq[i, t, -1] = t / float(seq_len - 1)
    return X_seq

seq_length = y_train_np.shape[1]
X_train_seq = create_sequence_inputs(X_train_np, seq_length)
X_val_seq = create_sequence_inputs(X_val_np, seq_length)
X_test_seq = create_sequence_inputs(X_test_np, seq_length)

# 转换为PyTorch张量
X_train_seq_tensor = torch.tensor(X_train_seq)
X_val_seq_tensor = torch.tensor(X_val_seq)
X_test_seq_tensor = torch.tensor(X_test_seq)
y_train_tensor = torch.tensor(y_train_np)
y_val_tensor = torch.tensor(y_val_np)
y_test_tensor = torch.tensor(y_test_np)

# 定义LSTM模型
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_len):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.output_len = output_len

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out).squeeze(-1)
        return out

# 初始化模型和优化器
input_dim = X_train_seq.shape[2]
hidden_dim = 64
lstm_model = LSTMRegressor(input_dim, hidden_dim, seq_length)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 训练参数设置
checkpoint_path = "./models/checkpoint_lstm.pth"
best_model_path = "./models/best_model_lstm.pth"
epochs = 50000
best_val_loss = float('inf')
start_epoch = 0

# 加载检查点（如果存在）
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    lstm_model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    print(f"从检查点恢复训练，起始轮次：{start_epoch}，当前最佳验证损失：{best_val_loss:.4f}")

# 准备数据加载器
train_dataset = TensorDataset(X_train_seq_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 训练循环
for epoch in range(start_epoch, epochs):
    # 训练阶段
    lstm_model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # 验证阶段
    lstm_model.eval()
    with torch.no_grad():
        val_preds = lstm_model(X_val_seq_tensor)
        val_loss = criterion(val_preds, y_val_tensor).item()
    
    # 保存最佳模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(lstm_model.state_dict(), best_model_path)
        print(f"发现新的最佳模型，验证损失：{val_loss:.4f}，模型已保存")
    
    # 保存检查点
    checkpoint = {
        'epoch': epoch,
        'model_state': lstm_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
    }
    torch.save(checkpoint, checkpoint_path)
    
    # 打印训练信息
    if (epoch + 1) % 10 == 0:
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] 训练损失：{avg_train_loss:.4f} 验证损失：{val_loss:.4f}")

# 最终测试
lstm_model.load_state_dict(torch.load(best_model_path))
lstm_model.eval()
with torch.no_grad():
    test_preds = lstm_model(X_test_seq_tensor)
    mse = mean_squared_error(y_test_np, test_preds.numpy())
    rmse = math.sqrt(mse)
    r2 = r2_score(y_test_np, test_preds.numpy())

print(f"\n最终测试结果：")
print(f"MSE: {mse:.4f}  RMSE: {rmse:.4f}  R²: {r2:.4f}")

从检查点恢复训练，起始轮次：64260，当前最佳验证损失：57.7219

最终测试结果：
MSE: 113.8639  RMSE: 10.6707  R²: 0.9092


/tmp/ipykernel_959952/3172726256.py:97: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)
/tmp/ipykernel_959952/3172726256.py:148: Futur